In [59]:
import urllib.request, json
import pandas as pd
from tqdm import tqdm

In [2]:
def get_json(url):
    with urllib.request.urlopen(url) as resp:
        return json.load(resp)

## Input params

In [47]:
url_link = r"https://aflapi.afl.com.au/afl/v2/compseasons/73/award/brownlow?page"
n_page = 27
size_pages = 30

In [45]:
brownlow_data_afl = get_json("https://aflapi.afl.com.au/afl/v2/compseasons/73/award/brownlow?page=1&pageSize=30")

## Helper function

In [41]:
def ensure_list_of_dicts(x):
    # parse whole-string JSON
    if isinstance(x, str):
        x = json.loads(x)
    # if top-level dict that contains the list we want, try to find it
    if isinstance(x, dict):
        # pick the first list-of-dicts value we find
        for v in x.values():
            if isinstance(v, list) and v and isinstance(v[0], dict):
                x = v
                break
        else:
            # single dict -> wrap it
            x = [x]
    if not isinstance(x, list):
        raise ValueError("Input not list/dict/JSON-string")
    out = []
    for item in x:
        if isinstance(item, dict):
            out.append(item)
        elif isinstance(item, str):
            try:
                parsed = json.loads(item)
            except Exception:
                # skip non-json strings
                continue
            if isinstance(parsed, dict):
                out.append(parsed)
            elif isinstance(parsed, list):
                out.extend([i for i in parsed if isinstance(i, dict)])
        # ignore anything else
    return out

In [50]:
page=5

test_url = f"{url_link}={page}&pageSize={size_pages}"

test_json = get_json(test_url)

In [55]:
def json_to_pandas_long(json_data):

    # ensure your variable is a list of dicts
    parsed_json = ensure_list_of_dicts(json_data)

    # now the original unpack logic (slightly compact)
    rows = []
    for p in parsed_json:
        base = {
            'id': p.get('id'),
            'playerProviderId': p.get('providerId'),
            'firstName': p.get('firstName'),
            'surname': p.get('surname'),
            'eligible': p.get('eligible'),
            'teamId': p.get('teamId'),
            'totalVotes': p.get('totalVotes'),
        }
        for round_str, entries in p.get('rounds', {}).items():
            for e in entries:
                r = base.copy()
                r['round'] = int(round_str)
                r['entryProviderId'] = e.get('providerId')
                r['played'] = e.get('played')
                r['bye'] = e.get('bye')
                r['points'] = pd.to_numeric(e.get('points'), errors='coerce')
                rows.append(r)

    long_df = pd.DataFrame(rows).sort_values(['id', 'round']).reset_index(drop=True)

    return long_df

In [60]:
scraped_list = []

for i in tqdm(range(n_page)):
    target_url = f"{url_link}={i}&pageSize={size_pages}"
    scraped_json = get_json(target_url)
    scraped_df = json_to_pandas_long(scraped_json)
    scraped_list.append(scraped_df)

  0%|          | 0/27 [00:00<?, ?it/s]

100%|██████████| 27/27 [00:35<00:00,  1.32s/it]


In [61]:
scraped_final_df = pd.concat(scraped_list)

In [62]:
scraped_final_df.sample(n=5)

,id,playerProviderId,firstName,surname,eligible,teamId,totalVotes,round,entryProviderId,played,bye,points
35,752,CD_I294859,Jeremy,McGovern,True,18,0,18,CD_M20250141801,False,None,NaN
132,1897,CD_I1008312,Liam,Stocker,True,11,0,17,CD_M20250141703,False,None,NaN
399,9439,CD_I1032119,James,Leake,True,15,0,10,CD_M20250141005,False,None,NaN
177,6489,CD_I1027687,Tom,Hanily,True,13,0,22,CD_M20250142209,False,None,NaN
174,6489,CD_I1027687,Tom,Hanily,True,13,0,19,CD_M20250141907,False,None,NaN


In [63]:
scraped_final_df.shape

(10371, 12)

In [64]:
scraped_final_df[['firstName','surname']].drop_duplicates()

,firstName,surname
0,Zach,Merrett
12,Jeremy,Cameron
22,Patrick,Cripps
31,Lachie,Neale
45,Max,Gawn
...,...,...
243,Matt,Whitlock
266,Jack,Whitlock
287,Tyler,Welsh
314,Charlie,West


In [68]:
summary_by_player = (scraped_final_df
                     .groupby(['firstName','surname','teamId'])['points'].sum()
                     .reset_index()
                     .sort_values(by='points', ascending=False)
                     .reset_index(drop=True))
summary_by_player.head(n=5)

,firstName,surname,teamId,points
0,Jordan,Dawson,1,32.0
1,Nick,Daicos,3,31.0
2,Bailey,Smith,10,31.0
3,Noah,Anderson,4,30.0
4,Hugh,McCluggage,2,28.0


In [76]:
output_df = (scraped_final_df
 .pivot_table('points',['firstName','surname','teamId'],'round', aggfunc='sum')
 .add_prefix('round_')
 .assign(total_votes = lambda x: x.sum(axis=1))
 .fillna(0)
 .sort_values(by='total_votes', ascending=False)
 .reset_index()
)


In [77]:
output_df.to_csv(r"C:\Users\mrjay\python\jupyter_notebooks\Projects\brownlow\data\afl_website_2025.csv")